# Part 1 — Sorting Algorithm Comparison
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---

### How to use this notebook
Run each cell top-to-bottom with **Shift+Enter**.  
No external widgets needed — everything outputs directly in the cell.

| Cell | What it does |
|------|-------------|
| **Cell 1** | All 8 algorithm definitions + complexity reference table |
| **Cell 2** | Step-by-step swap trace on a small custom array |
| **Cell 3** | Full benchmark — n=1000 by default, change `DATASET_SIZE` |
| **Cell 4** | Performance charts (bar, scalability curve, heatmap) |
| **Cell 5** | Automated test suite — verifies all algorithms |

---

### Complexity Reference

| Algorithm | Best | Average | Worst | Space | Stable |
|-----------|------|---------|-------|-------|--------|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | No |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | No |
| Random-Quick Sort | O(n log n) | O(n log n) | O(n²)* | O(log n) | No |
| Counting Sort | O(n+k) | O(n+k) | O(n+k) | O(k) | Yes |
| Radix Sort | O(nk) | O(nk) | O(nk) | O(n+k) | Yes |

*Worst-case extremely rare with random pivot.

---

### Flowcharts

**Bubble Sort**
```
for each pass i:
    for j in range(n - i - 1):
        if arr[j] > arr[j+1]:  ──► swap → swapped = True
    if not swapped: STOP EARLY  (array already sorted)
```

**Merge Sort**
```
merge_sort(arr):
    if len(arr) <= 1: return arr          ← base case
    left  = merge_sort(arr[:mid])         ← recurse left
    right = merge_sort(arr[mid:])         ← recurse right
    return merge(left, right)             ← O(n) merge step
```

**Quick Sort**
```
quick_sort(lo, hi):
    if lo >= hi: return                   ← base case
    pivot = arr[hi]                       ← last element (or random)
    partition: ≤pivot → left, >pivot → right
    recurse on left and right partitions
```

**Counting Sort**
```
count occurrences of each value in count[]
rebuild sorted array from count[]         ← no comparisons needed!
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 1 — ALL 8 SORTING ALGORITHMS
#  Data structure: Python list (dynamic array, O(1) random access)
#  Each function returns: (sorted_list, comparisons, swaps)
#  No external libraries needed — pure Python throughout.
# ═══════════════════════════════════════════════════════════════════════
import time
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    "figure.dpi": 110,
    "font.family": "monospace",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
})
COLORS = ["#E63946","#457B9D","#2A9D8F","#E9C46A","#F4A261","#264653","#A8DADC","#9B5DE5"]

random.seed(42)
np.random.seed(42)


# ── BUBBLE SORT ────────────────────────────────────────────────────────
# Strategy  : Compare adjacent pairs; swap if out of order.
# Key opt.  : Early-exit when a full pass makes zero swaps.
# Data str. : In-place array, two index pointers j and j+1.
def bubble_sort(arr):
    a = arr[:]
    n = len(a)
    comps = swaps = 0
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):        # inner range shrinks each pass
            comps += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swaps += 1
                swapped = True
        if not swapped:
            break                          # already sorted — stop early
    return a, comps, swaps


# ── SELECTION SORT ─────────────────────────────────────────────────────
# Strategy  : Each pass finds the minimum; places it at front.
# Note      : Always exactly n(n-1)/2 comparisons — good when writes cost.
def selection_sort(arr):
    a = arr[:]
    n = len(a)
    comps = swaps = 0
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            comps += 1
            if a[j] < a[min_idx]:
                min_idx = j
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]
            swaps += 1
    return a, comps, swaps


# ── INSERTION SORT ─────────────────────────────────────────────────────
# Strategy  : Maintain sorted prefix; insert each element in place.
# Analogy   : Sorting playing cards in your hand.
# Best case : O(n) on nearly-sorted input.
def insertion_sort(arr):
    a = arr[:]
    comps = swaps = 0
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0:
            comps += 1
            if a[j] > key:
                a[j + 1] = a[j]
                swaps += 1
                j -= 1
            else:
                break
        a[j + 1] = key
    return a, comps, swaps


# ── MERGE SORT ─────────────────────────────────────────────────────────
# Strategy  : Divide-and-conquer. Split → sort halves → merge.
# Data str. : Auxiliary arrays created at each merge step.
# Guarantee : O(n log n) on ALL inputs — no worst-case scenario.
def merge_sort(arr):
    comps = [0]

    def merge(L, R):
        out = []; i = j = 0
        while i < len(L) and j < len(R):
            comps[0] += 1
            if L[i] <= R[j]: out.append(L[i]); i += 1
            else:             out.append(R[j]); j += 1
        out.extend(L[i:]); out.extend(R[j:])
        return out

    def sort(a):
        if len(a) <= 1: return a
        m = len(a) // 2
        return merge(sort(a[:m]), sort(a[m:]))

    return sort(arr[:]), comps[0], 0


# ── QUICK SORT ─────────────────────────────────────────────────────────
# Strategy  : Choose pivot; partition ≤pivot left, >pivot right; recurse.
# Worst case: O(n²) when pivot is always min/max (sorted input + last pivot).
def quick_sort(arr):
    comps = [0]; swaps = [0]

    def partition(a, lo, hi):
        pivot = a[hi]; i = lo - 1
        for j in range(lo, hi):
            comps[0] += 1
            if a[j] <= pivot:
                i += 1; a[i], a[j] = a[j], a[i]; swaps[0] += 1
        a[i + 1], a[hi] = a[hi], a[i + 1]; swaps[0] += 1
        return i + 1

    def sort(a, lo, hi):
        if lo < hi:
            p = partition(a, lo, hi)
            sort(a, lo, p - 1); sort(a, p + 1, hi)

    a = arr[:]
    sort(a, 0, len(a) - 1)
    return a, comps[0], swaps[0]


# ── RANDOMIZED QUICK SORT ──────────────────────────────────────────────
# Same as Quick Sort but pivot chosen randomly — eliminates O(n²) worst case.
# Swap random element to end before partitioning.
def random_quick_sort(arr):
    comps = [0]; swaps = [0]

    def partition(a, lo, hi):
        r = random.randint(lo, hi)
        a[r], a[hi] = a[hi], a[r]; swaps[0] += 1
        pivot = a[hi]; i = lo - 1
        for j in range(lo, hi):
            comps[0] += 1
            if a[j] <= pivot:
                i += 1; a[i], a[j] = a[j], a[i]; swaps[0] += 1
        a[i + 1], a[hi] = a[hi], a[i + 1]; swaps[0] += 1
        return i + 1

    def sort(a, lo, hi):
        if lo < hi:
            p = partition(a, lo, hi)
            sort(a, lo, p - 1); sort(a, p + 1, hi)

    a = arr[:]
    sort(a, 0, len(a) - 1)
    return a, comps[0], swaps[0]


# ── COUNTING SORT ──────────────────────────────────────────────────────
# Non-comparison sort for non-negative integers.
# Data str. : Frequency count array of size k = max_value + 1.
# Breaks the O(n log n) lower bound for comparison-based sorts.
def counting_sort(arr):
    if not arr: return [], 0, 0
    a   = arr[:]
    mx  = max(a)
    cnt = [0] * (mx + 1)
    comps = 0
    for v in a:
        cnt[v] += 1
        comps  += 1                  # count assignment as "comparison"
    result = [i for i, c in enumerate(cnt) for _ in range(c)]
    return result, comps, 0


# ── RADIX SORT (LSD) ───────────────────────────────────────────────────
# Non-comparison sort. Sorts digit by digit (least to most significant).
# Data str. : 10 buckets (lists), one per digit 0–9.
def radix_sort(arr):
    if not arr: return [], 0, 0
    a   = arr[:]
    mx  = max(a)
    comps = 0
    exp = 1
    while mx // exp > 0:
        buckets = [[] for _ in range(10)]
        for num in a:
            buckets[(num // exp) % 10].append(num)
            comps += 1
        a   = [x for b in buckets for x in b]
        exp *= 10
    return a, comps, 0


# ── ALGORITHM REGISTRY ─────────────────────────────────────────────────
ALGORITHMS = [
    ("Bubble Sort",       bubble_sort),
    ("Selection Sort",    selection_sort),
    ("Insertion Sort",    insertion_sort),
    ("Merge Sort",        merge_sort),
    ("Quick Sort",        quick_sort),
    ("Random-Quick Sort", random_quick_sort),
    ("Counting Sort",     counting_sort),
    ("Radix Sort",        radix_sort),
]

SHORT = ["Bubble","Selection","Insertion","Merge","Quick","Rnd-Quick","Counting","Radix"]

print("=" * 52)
print("  Part 1 — Sorting Algorithms  |  8 loaded")
print("=" * 52)
for (name, _), col in zip(ALGORITHMS, COLORS):
    print(f"  ●  {name}")
print()
print("  Run the next cells to benchmark, trace, and test.")


# ── DATASET GENERATOR ─────────────────────────────────────────────────
# Used by benchmark, trace, and charts cells.
# Change INPUT_TYPE: "random" | "sorted" | "reversed"
def make_dataset(n, itype="random"):
    if itype == "random":
        return [random.randint(0, 9999) for _ in range(n)]
    elif itype == "sorted":
        return list(range(n))
    elif itype == "reversed":
        return list(range(n, 0, -1))
    else:
        raise ValueError(f"Unknown itype: {itype!r}. Use 'random', 'sorted', or 'reversed'")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 2 — STEP-BY-STEP TRACE
#  Shows every swap Bubble Sort makes on a small array.
#  Change INPUT_ARRAY to trace any values you like.
# ═══════════════════════════════════════════════════════════════════════

INPUT_ARRAY = [9, 3, 7, 1, 5, 8, 2, 6, 4]   # ← change me!

# ── Bubble sort trace ──────────────────────────────────────────────────
def trace_bubble(arr):
    a = arr[:]
    n = len(a)
    step = total_comps = total_swaps = 0

    print("─" * 58)
    print(f"  BUBBLE SORT — Swap-by-Swap Trace")
    print("─" * 58)
    print(f"  Input : {a}")
    print()

    for i in range(n):
        pass_swaps = 0
        for j in range(n - i - 1):
            total_comps += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                step        += 1
                pass_swaps  += 1
                total_swaps += 1
                print(f"  Step {step:>3}  swap[{j}]↔[{j+1}]"
                      f"  →  {a}")
        if pass_swaps == 0:
            print(f"  Pass {i+1:>2}  — zero swaps → EARLY EXIT")
            break
        else:
            print(f"  ── end of pass {i+1} ({pass_swaps} swap{'s' if pass_swaps>1 else ''}) ──")
    print()
    print(f"  Sorted : {a}")
    print(f"  Total comparisons : {total_comps}")
    print(f"  Total swaps       : {total_swaps}")
    print("─" * 58)

trace_bubble(INPUT_ARRAY)

# ── Also trace Insertion Sort for contrast ─────────────────────────────
print()
def trace_insertion(arr):
    a = arr[:]
    step = 0
    print("─" * 58)
    print(f"  INSERTION SORT — Insert-by-Insert Trace")
    print("─" * 58)
    print(f"  Input : {a}")
    print()
    for i in range(1, len(a)):
        key = a[i]; j = i - 1; moved = 0
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]; j -= 1; moved += 1
        a[j + 1] = key
        step += 1
        action = f"insert {key} at [{j+1}]" if moved else f"{key} already in place"
        print(f"  Step {step:>2}  {action:<25}  →  {a}")
    print()
    print(f"  Sorted : {a}")
    print("─" * 58)

trace_insertion(INPUT_ARRAY)


In [ ]:
# =========================================================
#  CELL 3 - BENCHMARK
#
#  HOW TO CHOOSE YOUR DATASET:
#    1. Change DATASET_SIZE to any positive integer (default: 1000)
#    2. Change INPUT_TYPE to one of:
#         "random"   - random integers (average case)
#         "sorted"   - already sorted  (best case for bubble/insertion)
#         "reversed" - reverse sorted  (worst case for most algorithms)
#    3. Run this cell (Shift+Enter)
#
#  The spec default is n=1000 with random input.
# =========================================================

# ── USER INPUTS — change these and re-run ─────────────────
DATASET_SIZE = 1000           # try: 100, 500, 1000, 2000, 5000
INPUT_TYPE   = "random"       # "random" | "sorted" | "reversed"
# ──────────────────────────────────────────────────────────

# Input validation
VALID_TYPES = ("random", "sorted", "reversed")
if not isinstance(DATASET_SIZE, int) or DATASET_SIZE < 2:
    raise ValueError(f"DATASET_SIZE must be an integer >= 2, got {DATASET_SIZE!r}")
if INPUT_TYPE not in VALID_TYPES:
    raise ValueError(f"INPUT_TYPE must be one of {VALID_TYPES}, got {INPUT_TYPE!r}")

random.seed(0)
dataset  = make_dataset(DATASET_SIZE, INPUT_TYPE)
expected = sorted(dataset)

print("=" * 68)
print(f"  BENCHMARK  |  Dataset Size: {DATASET_SIZE}  |  Input: {INPUT_TYPE}")
print("=" * 68)
print(f"  Input (first 15)  : {dataset[:15]}...")
print()

results = []
for name, fn in ALGORITHMS:
    t0 = time.perf_counter()
    sorted_out, comps, swaps = fn(dataset[:])
    elapsed_ms = (time.perf_counter() - t0) * 1000
    correct    = (sorted_out == expected)
    results.append((name, elapsed_ms, comps, swaps, sorted_out, correct))

# Sort by time — fastest first
results.sort(key=lambda r: r[1])

# ── Spec-format table ──────────────────────────────────────
print(f"  Dataset Size: {DATASET_SIZE}")
print()
W = [22, 11, 15, 12, 6]
header = (f"  {'Algorithm':<{W[0]}} {'Time (ms)':>{W[1]}}"
          f"  {'Comparisons':>{W[2]}} {'Swaps':>{W[3]}} {'OK?':>{W[4]}}")
print(header)
print("  " + "-" * (sum(W) + 8))
for rank, (name, ms, comps, swaps, _, correct) in enumerate(results, 1):
    crown = "  <- FASTEST" if rank == 1 else ""
    ok    = "OK" if correct else "FAIL"
    print(f"  {name:<{W[0]}} {ms:>{W[1]}.3f}"
          f"  {comps:>{W[2]},} {swaps:>{W[3]},} {ok:>{W[4]}}{crown}")

print()
fastest = results[0]; slowest = results[-1]
print(f"  Fastest : {fastest[0]}  ({fastest[1]:.3f} ms)")
print(f"  Slowest : {slowest[0]}  ({slowest[1]:.3f} ms)")
if fastest[1] > 0:
    print(f"  Ratio   : {slowest[1]/fastest[1]:.1f}x slower")
print()
s = results[0][4]
print(f"  Sorted Output (first 20 elements):")
print(f"  {s[:20]}")
print("=" * 68)


In [ ]:
# =========================================================
#  CELL 4 - PERFORMANCE CHARTS
#  Chart A : Benchmark bars (time / comparisons / swaps)
#  Chart B : Scalability curve
#  Chart C : Best / Average / Worst heatmap
# =========================================================

# Re-run benchmark silently for fresh data
import numpy as _np2
random.seed(0)
_data = make_dataset(DATASET_SIZE, "random")
_res  = []
for _name, _fn in ALGORITHMS:
    _t0 = time.perf_counter()
    _, _c, _s = _fn(_data[:])
    _res.append((_name, (time.perf_counter()-_t0)*1000, _c, _s))

_names_s  = [r[0].replace(" Sort","").replace("-"," ") for r in _res]
_times    = [r[1] for r in _res]
_comps    = [r[2] for r in _res]
_swaps    = [r[3] for r in _res]

# -- Chart A: Benchmark Bars ------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Sorting Benchmark  (n = {DATASET_SIZE})",
             fontsize=13, fontweight="bold", y=1.01)

for ax, vals, fmt, title in zip(
        axes,
        [_times, _comps, _swaps],
        [".3f", ",d", ",d"],
        ["Execution Time (ms)", "Comparisons", "Swaps"]):
    pairs = sorted(zip(vals, _names_s, COLORS), reverse=True)
    v, l, c = zip(*pairs)
    bars = ax.barh(l, v, color=c, edgecolor="white", lw=1.2, height=0.6)
    ax.bar_label(bars,
                 labels=[format(x, fmt) for x in v],
                 padding=4, fontsize=8)
    ax.set_title(title, pad=8)
    ax.set_xlabel("ms" if fmt==".3f" else "count")
    ax.spines[["top","right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()
plt.savefig("/tmp/p1_chart_a.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart A: benchmark bars - done")


# -- Chart B: Scalability ----------------------------------------------
SIZES = [50, 100, 250, 500, 1000, 2000]
scale = {name: [] for name, _ in ALGORITHMS}
print("\nRunning scalability test...")
for sz in SIZES:
    d = [random.randint(0, 9999) for _ in range(sz)]
    for name, fn in ALGORITHMS:
        t0 = time.perf_counter()
        fn(d[:])
        scale[name].append((time.perf_counter() - t0) * 1000)
    print(f"  n={sz:>5} done")

fig2, ax2 = plt.subplots(figsize=(11, 5))
for i, (name, _) in enumerate(ALGORITHMS):
    ax2.plot(SIZES, scale[name], marker="o", label=name,
             color=COLORS[i], lw=2, ms=6, alpha=0.9)
ax2.set_xlabel("Dataset Size  (n)")
ax2.set_ylabel("Time  (ms)")
ax2.set_title("Scalability - Execution Time vs Dataset Size", pad=10)
ax2.legend(loc="upper left", fontsize=9, framealpha=0.9,
           ncol=2, columnspacing=0.8)
ax2.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/tmp/p1_chart_b.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart B: scalability - done")


# -- Chart C: Heatmap --------------------------------------------------
n_h = 500
cases = {
    "Best (sorted)":    list(range(n_h)),
    "Average (random)": [random.randint(0,9999) for _ in range(n_h)],
    "Worst (reversed)": list(range(n_h, 0, -1)),
}
matrix = []
for name, fn in ALGORITHMS:
    row = []
    for d in cases.values():
        t0 = time.perf_counter()
        fn(d[:])
        row.append((time.perf_counter()-t0)*1000)
    matrix.append(row)
M = _np2.array(matrix)

fig3, ax3 = plt.subplots(figsize=(9, 6))
im = ax3.imshow(M, cmap="YlOrRd", aspect="auto")
ax3.set_xticks(range(len(cases)))
ax3.set_xticklabels(list(cases.keys()), fontsize=10)
ax3.set_yticks(range(len(ALGORITHMS)))
ax3.set_yticklabels([n for n, _ in ALGORITHMS], fontsize=9)
for i in range(len(ALGORITHMS)):
    for j in range(len(cases)):
        v   = M[i, j]
        col = "white" if v > M.max() * 0.55 else "black"
        ax3.text(j, i, f"{v:.2f}", ha="center", va="center",
                 color=col, fontsize=9, fontweight="bold")
plt.colorbar(im, ax=ax3, label="Time (ms)", shrink=0.8)
ax3.set_title(f"Best / Average / Worst Case Heatmap  (n={n_h})", pad=10)
plt.tight_layout()
plt.savefig("/tmp/p1_chart_c.png", bbox_inches="tight", dpi=110)
plt.show()
print("Chart C: heatmap - done")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 5 — AUTOMATED TEST SUITE
#  Verifies all 8 algorithms on a range of edge cases and large inputs.
#  Passes when output matches Python's built-in sorted() exactly.
# ═══════════════════════════════════════════════════════════════════════

TEST_CASES = [
    ("Empty array",          []),
    ("Single element",       [42]),
    ("Two — ordered",        [1, 2]),
    ("Two — reversed",       [9, 1]),
    ("All duplicates",       [5, 5, 5, 5, 5]),
    ("Already sorted",       list(range(10))),
    ("Reverse sorted",       list(range(9, -1, -1))),
    ("With duplicates",      [3,1,4,1,5,9,2,6,5,3,5]),
    ("Random n=50",          [random.randint(0,999) for _ in range(50)]),
    ("Random n=200",         [random.randint(0,9999) for _ in range(200)]),
    ("Random n=1000",        [random.randint(0,9999) for _ in range(1000)]),
]

print("=" * 65)
print("  AUTOMATED TEST SUITE  —  All 8 Algorithms")
print("=" * 65)

total = passed = 0
all_pass = True

for algo_name, fn in ALGORITHMS:
    fails = []
    for tc_name, tc_data in TEST_CASES:
        total += 1
        try:
            result, _, _ = fn(tc_data[:])
            if result == sorted(tc_data):
                passed += 1
            else:
                fails.append(tc_name); all_pass = False
        except Exception as e:
            fails.append(f"{tc_name} [ERR:{e}]"); all_pass = False

    status = "✓  ALL PASS" if not fails else f"✗  FAILED: {fails}"
    print(f"  {algo_name:<22}  {len(TEST_CASES) - len(fails)}/{len(TEST_CASES)}  {status}")

print("─" * 65)
print(f"  Total: {passed}/{total} tests passed")
if all_pass:
    print("  🎉 All algorithms verified correct!")
print("=" * 65)
